# Prepare environment

In [ ]:
%reload_ext autoreload
%autoreload 2

import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

# Load data

In [ ]:
import json
import os

import torch
from torch.utils.data import DataLoader
from torchvision.transforms import v2

from src.dataset import TestDataset, TrainDataset

cfg_path = "../cfg.json"
print(f"Loading configuration from {cfg_path}")
with open(cfg_path, "r") as f:
    cfg = json.load(f)

data_dir = os.path.join("../", cfg["data"]["dir"])
batch_size = cfg["data"]["batch_size"]
num_workers = cfg["data"]["num_workers"]
train_ratio = cfg["data"]["train_ratio"]
print("Data configuration:")
print(f"Data directory: {data_dir}")
print(f"Batch size: {batch_size}")
print(f"Number of workers: {num_workers}")
print(f"Train ratio: {train_ratio}")

transform = v2.Compose([
    v2.Normalize((0.5,), (0.5,))
])

full_dataset = TrainDataset(data_dir, transform=transform)

train_size = int(len(full_dataset) * train_ratio)
valid_size = len(full_dataset) - train_size

train_dataset, valid_dataset = torch.utils.data.random_split(full_dataset, [train_size, valid_size])
test_dataset = TestDataset(data_dir, transform=transform)

print(f"Full dataset size: {len(full_dataset)}")
print(f"Train dataset size: {len(train_dataset)}")
print(f"Validation dataset size: {len(valid_dataset)}")
print(f"Test dataset size: {len(test_dataset)}")

train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=num_workers)
valid_dataloader = DataLoader(valid_dataset, batch_size=batch_size, shuffle=True, num_workers=num_workers)
test_dataloader = DataLoader(test_dataset, batch_size=batch_size, shuffle=True, num_workers=num_workers)

# Visualize a data sample

In [ ]:
%matplotlib inline

import matplotlib.pyplot as plt
import torch

features, labels = next(iter(train_dataloader))
print(f"Using features of shape: {features.shape}")
print(f"Using labels of shape: {labels.shape}")
print(
    "Batch statistics: "
    + f"mean {features.mean(dim=[0, 2, 3]).item():0.3f}, "
    + f"std {features.std(dim=[0, 2, 3]).item():0.3f}, "
    + f"min {torch.amin(features, dim=(0, 2, 3)).item():0.3f}, "
    + f"max {torch.amax(features, dim=(0, 2, 3)).item():0.3f}"
)  

num_images = 9
num_images = min(num_images, batch_size)

num_rows = int(torch.sqrt(torch.tensor(num_images)))
num_cols = int(torch.sqrt(torch.tensor(num_images)))
fig, axes = plt.subplots(num_rows, num_cols, figsize=(2*num_cols, 2*num_rows))

for ax, feature, label in zip(axes.flat, features[:num_images], labels[:num_images]):
    ax.imshow(feature.squeeze(), cmap="gray")
    ax.set_title(str(label.item()))
    ax.axis("off")

plt.tight_layout()